# 03 — Topic Analysis

Runs the topics question over entries (resumable), then plots topic frequency.

**Smoke mode:** Runs over the most recent `LIMIT` entries. Set `LIMIT = None` for full corpus.

In [1]:
import sys, json
sys.path.insert(0, "../src")

import pandas as pd
import plotly.express as px

from journal.config import LANCE_ROOT, LLM_MODEL
from journal.analyze import BatchAnalyzer, QuestionRegistry
from journal.llm import OllamaLLM
from journal.questions.topics import TOPICS
from journal.store import Store

LIMIT = 200

store = Store(LANCE_ROOT)

entries_df = store.entries_to_pandas()
if LIMIT is not None:
    entries_df = entries_df.sort_values("date").tail(LIMIT)
target_ids = set(entries_df["id"].tolist())
print(f"target entries: {len(target_ids)}")

original_missing = store.entries_missing_analysis
def scoped_missing(question_id, model):
    all_missing = original_missing(question_id, model)
    return [eid for eid in all_missing if eid in target_ids]
store.entries_missing_analysis = scoped_missing

reg = QuestionRegistry()
reg.register(TOPICS)
analyzer = BatchAnalyzer(store=store, llm=OllamaLLM(), registry=reg, model=LLM_MODEL, max_workers=4)
report = analyzer.run("topics")
print(report)

target entries: 200
AnalyzeReport(question_id='topics', processed=0, failed=0, errors=[])


In [2]:
adf = store.analyses_to_pandas()
adf = adf[(adf["question_id"] == "topics") & adf["parsed_ok"]].copy()
adf["topics"] = adf["result_json"].apply(lambda s: json.loads(s)["topics"])
adf.head()

,entry_id,question_id,question_text,result_json,parsed_ok,model,created_at,topics
200,c4f6d3864e7b24de,topics,Identify the top 3 topics in this journal entr...,"{""topics"": [""work"", ""trump"", ""mamadadu""]}",True,gemma3:4b,2026-06-30 22:53:31.475277,"[work, trump, mamadadu]"
201,62a2339631e6adf1,topics,Identify the top 3 topics in this journal entr...,"{""topics"": [""switch 2"", ""train cancellations"",...",True,gemma3:4b,2026-06-30 22:53:32.989503,"[switch 2, train cancellations, jiarong]"
202,fbf397b1f4b7aa51,topics,Identify the top 3 topics in this journal entr...,"{""topics"": [""pokemon game"", ""ramen dinner"", ""f...",True,gemma3:4b,2026-06-30 22:53:35.051998,"[pokemon game, ramen dinner, family conflict]"
203,b15e08a513844d19,topics,Identify the top 3 topics in this journal entr...,"{""topics"": [""switch console"", ""guilt"", ""loneli...",True,gemma3:4b,2026-06-30 22:53:35.818248,"[switch console, guilt, loneliness]"
204,5b3ec6b02347706e,topics,Identify the top 3 topics in this journal entr...,"{""topics"": [""grading assignments"", ""exam prepa...",True,gemma3:4b,2026-06-30 22:53:36.874342,"[grading assignments, exam preparation, relati..."


In [3]:
exploded = adf.explode("topics")
top = exploded["topics"].value_counts().head(30).reset_index()
top.columns = ["topic", "count"]
fig = px.bar(top, x="count", y="topic", orientation="h",
             title="Top 30 topics across all entries")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

In [4]:
edf = store.entries_to_pandas()[["id", "date"]]
m = adf.merge(edf, left_on="entry_id", right_on="id")
m["date"] = pd.to_datetime(m["date"])
m = m.explode("topics")
top_topics = m["topics"].value_counts().head(8).index.tolist()
sub = m[m["topics"].isin(top_topics)]
freq = sub.groupby([sub["date"].dt.to_period("M"), "topics"]).size().reset_index(name="count")
freq["date"] = freq["date"].dt.to_timestamp()
fig = px.line(freq, x="date", y="count", color="topics", title="Top topics over time (monthly)")
fig.show()